# Email Spam Detection Using Machine Learning
**Google Colab notebook** — TF-IDF + Multinomial Naive Bayes

This notebook follows the project brief: validate the input data, inspect data quality, train a baseline text classifier, evaluate precision/recall/F1 and false positives, and test custom email text.

**Dataset note:** `email_spam_dataset_500.csv` is a synthetic teaching dataset created for this project (250 spam-like and 250 legitimate examples). It is not a real-world email corpus. Results on it are demonstrations only and should not be presented as real-world accuracy.

## 1. Upload the dataset
Run this cell and upload `email_spam_dataset_500.csv` (or the `.xlsx` file).

In [ ]:
from google.colab import files
uploaded = files.upload()

# Prefer the CSV file; Excel is also supported.
import pandas as pd
from io import BytesIO

filename = next(iter(uploaded))
if filename.lower().endswith(".csv"):
    df = pd.read_csv(BytesIO(uploaded[filename]))
elif filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(BytesIO(uploaded[filename]))
else:
    raise ValueError("Unsupported file. Please upload a CSV or Excel file.")

print("Loaded:", filename)
print("Rows and columns:", df.shape)
df.head()

## 2. Validate and prepare data
Required columns are `text` and `label`. Labels should be `spam` or `ham`.

In [ ]:
required_columns = {"text", "label"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["label"] = df["label"].fillna("").astype(str).str.strip().str.lower()

before = len(df)
df = df[(df["text"] != "") & (df["label"].isin(["spam", "ham"]))]
removed = before - len(df)
duplicate_count = int(df.duplicated(subset=["text", "label"]).sum())
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

if df["label"].nunique() < 2:
    raise ValueError("Dataset must contain both 'spam' and 'ham' labels.")
if len(df) < 10:
    raise ValueError("Please provide at least 10 valid rows.")

print(f"Invalid/empty rows removed: {removed}")
print(f"Duplicate rows removed: {duplicate_count}")
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nMissing values after preparation:")
print(df.isna().sum())

## 3. Split, vectorise and train
TF-IDF converts text into numerical features. The vectoriser is fitted on training data only to avoid test-data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.20, random_state=42, stratify=df["label"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))),
    ("classifier", MultinomialNB())
])
model.fit(X_train, y_train)
print("Training completed.")
print(f"Training examples: {len(X_train)}")
print(f"Test examples: {len(X_test)}")

## 4. Evaluate model
Accuracy is included for context, but precision, recall, F1-score and false positives are especially important for spam filtering.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print("Classification report:")
print(classification_report(y_test, y_pred, labels=["ham", "spam"], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
print("Confusion matrix (rows=actual [ham, spam], columns=predicted [ham, spam]):")
print(cm)

# False positives: legitimate (ham) messages incorrectly marked as spam.
false_positives = X_test[(y_test == "ham") & (pd.Series(y_pred, index=y_test.index) == "spam")]
print(f"\nFalse positives (ham predicted as spam): {len(false_positives)}")
for msg in false_positives.head(5):
    print("-", msg)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Confusion Matrix")
plt.show()

## 5. Test your own email
Enter a message to get a prediction and model probability. This is a prototype, not a guarantee that an email is safe or spam.

In [ ]:
from IPython.display import display
import ipywidgets as widgets

email_box = widgets.Textarea(
    value="Congratulations! You won a free gift card. Claim now!",
    placeholder="Paste email text here...",
    description="Email:",
    layout=widgets.Layout(width="100%", height="120px")
)
predict_button = widgets.Button(description="Check email", button_style="primary")
output = widgets.Output()

def predict_email(_):
    with output:
        output.clear_output()
        text_input = email_box.value.strip()
        if not text_input:
            print("Please enter some email text first.")
            return
        prediction = model.predict([text_input])[0]
        probabilities = model.predict_proba([text_input])[0]
        classes = model.named_steps["classifier"].classes_
        prob_map = dict(zip(classes, probabilities))
        print("Prediction:", "SPAM" if prediction == "spam" else "LEGITIMATE (HAM)")
        print(f"Model score for spam: {prob_map.get('spam', 0):.1%}")
        print("Note: predictions can be wrong, especially for messages unlike the training examples.")

predict_button.on_click(predict_email)
display(email_box, predict_button, output)

## 6. Export predictions (optional)
This creates a CSV containing the prepared messages and model predictions. Use only data you are authorised to process.

In [ ]:
results = df.copy()
results["predicted_label"] = model.predict(results["text"])
results.to_csv("email_spam_predictions.csv", index=False)
print("Saved email_spam_predictions.csv")
from google.colab import files
files.download("email_spam_predictions.csv")

## 7. Documented checks
1. Required `text` and `label` columns are checked.
2. Unsupported file extensions are rejected.
3. Empty text rows are removed.
4. Missing text/label values are handled.
5. Labels outside `spam` and `ham` are removed.
6. Duplicate text-label rows are removed.
7. Training requires both classes.
8. Very small datasets are rejected.
9. Train/test split is stratified and reproducible (`random_state=42`).
10. Empty custom email input is handled.

## Limitations
- The included dataset is synthetic and template-based; evaluation scores may look unusually high and do not reflect real-world performance.
- Real emails can contain context, obfuscation, multilingual text, images, or malicious links not represented here.
- Do not use the prediction alone to decide whether to open links or attachments.
- For a stronger project, replace the sample data with an appropriately licensed real dataset, document its source, and re-evaluate on a held-out test set.